# Operational schema — execution evidence

Proves the modeled operational schema (related tables + primary/foreign keys) exists and is live on the `geniebandits-dev` Lakebase branch. Outputs below are real query results captured at execution time.

In [1]:
import sys, textwrap
sys.path.insert(0, ".")
from lib import lb
conn = lb.connect("geniebandits-dev")
conn.autocommit = True
cur = conn.cursor()

def show(sql, title=None):
    if title: print(f"### {title}")
    print(textwrap.dedent(sql).strip())
    cur.execute(sql)
    if cur.description:
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        print("-> " + " | ".join(cols))
        for r in rows:
            print("   " + " | ".join(str(x) for x in r))
        print(f"({len(rows)} row(s))\n")
    else:
        print(f"-> OK ({cur.rowcount} affected)\n")

In [2]:
show("""SELECT table_name FROM information_schema.tables
        WHERE table_schema='northpeak_ops' ORDER BY table_name""",
     "Tables in northpeak_ops")

### Tables in northpeak_ops
SELECT table_name FROM information_schema.tables
        WHERE table_schema='northpeak_ops' ORDER BY table_name
-> table_name
   action_status_history
   products
   recovery_actions
   stores
(4 row(s))



In [3]:
show("""SELECT tc.table_name, kcu.column_name AS pk_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu USING (constraint_name, table_schema)
        WHERE tc.constraint_type='PRIMARY KEY' AND tc.table_schema='northpeak_ops'
        ORDER BY 1,2""", "Primary keys")

### Primary keys
SELECT tc.table_name, kcu.column_name AS pk_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu USING (constraint_name, table_schema)
        WHERE tc.constraint_type='PRIMARY KEY' AND tc.table_schema='northpeak_ops'
        ORDER BY 1,2


-> table_name | pk_column
   action_status_history | history_id
   products | product_id
   recovery_actions | action_id
   stores | store_id
(4 row(s))



In [4]:
show("""SELECT tc.table_name, kcu.column_name,
               ccu.table_name AS references_table, ccu.column_name AS references_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu USING (constraint_name, table_schema)
        JOIN information_schema.constraint_column_usage ccu
          ON ccu.constraint_name=tc.constraint_name AND ccu.table_schema=tc.table_schema
        WHERE tc.constraint_type='FOREIGN KEY' AND tc.table_schema='northpeak_ops'
        ORDER BY 1,2""", "Foreign keys (related tables)")

### Foreign keys (related tables)
SELECT tc.table_name, kcu.column_name,
               ccu.table_name AS references_table, ccu.column_name AS references_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu USING (constraint_name, table_schema)
        JOIN information_schema.constraint_column_usage ccu
          ON ccu.constraint_name=tc.constraint_name AND ccu.table_schema=tc.table_schema
        WHERE tc.constraint_type='FOREIGN KEY' AND tc.table_schema='northpeak_ops'
        ORDER BY 1,2


-> table_name | column_name | references_table | references_column
   action_status_history | action_id | recovery_actions | action_id
   recovery_actions | product_id | products | product_id
   recovery_actions | source_store_id | stores | store_id
   recovery_actions | store_id | stores | store_id
   recovery_actions | substitute_product_id | products | product_id
(5 row(s))



In [5]:
show("""SELECT 'stores' t, count(*) n FROM northpeak_ops.stores
        UNION ALL SELECT 'products', count(*) FROM northpeak_ops.products
        UNION ALL SELECT 'recovery_actions', count(*) FROM northpeak_ops.recovery_actions
        UNION ALL SELECT 'action_status_history', count(*) FROM northpeak_ops.action_status_history
        ORDER BY 1""", "Row counts")

### Row counts
SELECT 'stores' t, count(*) n FROM northpeak_ops.stores
        UNION ALL SELECT 'products', count(*) FROM northpeak_ops.products
        UNION ALL SELECT 'recovery_actions', count(*) FROM northpeak_ops.recovery_actions
        UNION ALL SELECT 'action_status_history', count(*) FROM northpeak_ops.action_status_history
        ORDER BY 1


-> t | n
   action_status_history | 3
   products | 200
   recovery_actions | 6
   stores | 400
(4 row(s))

